# generating images
Generates 'blurried' faces by sampling from the latent space of the trained VanillaVAE: z ~ N(0, I) -> decode(z).



In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
from PIL import Image

import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/tesi/" if 'google.colab' in str(get_ipython()) else "."
PYTORCH_VAE_REPO = os.path.join(BASE_DIR, 'src/PyTorch-VAE-master')
CHECKPOINT_PATH = os.path.join(BASE_DIR, 'trained_models', 'vae','vae_blur_checkpoint_16x16.pt')
OUTPUT_DIR = os.path.join(BASE_DIR, 'data', 'generated_images')

NUM_SAMPLES = 5000        # how many samples to produce
OUT_SIZE = 128             # final images resolution
BLOCK_SIZE = 8              # dimension of the JPEG blocks (8x8)
                        # OUT_SIZE must be multiple of BLOCK_SIZE (128 = 16*8)
DC_BLOCK_MODE = True        # True: each pixel in the grid 16x16 becomes the constant value
                            # (= just DC) of a 8x8 block
                            # False: resize bilinear
BATCH_SIZE = 64
TEMPERATURE = 1.3          # >1 = z samples from N(0, T^2*I) instead of N(0,I): more diversity
                            # 1.0 = original
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
sys.path.append(PYTORCH_VAE_REPO)
from models.vanilla_vae import VanillaVAE


def load_vae(checkpoint_path, device):
    ckpt = torch.load(checkpoint_path, map_location=device)
    hidden_dims = ckpt.get('hidden_dims', [128, 256, 512])
    model = VanillaVAE(in_channels=3, latent_dim=ckpt['latent_dim'],
                        hidden_dims=list(hidden_dims)).to(device)
    model.load_state_dict(ckpt['model'])
    model.eval()
    return model


def vae_generate(num_samples, vae, device, temperature=1.1):
    """z ~ N(0, temperature^2 * I) -> decode(z).
    No real image in input.
    temperature > 1 : more diversity
    Output: [0,1] float32 clamped, shape [num_samples, 3, 16, 16]
    """
    z = torch.randn(num_samples, vae.latent_dim, device=device) * temperature
    with torch.no_grad():
        samples = vae.decode(z)
    return samples.clamp(0.0, 1.0)


def to_dc_blocks(samples, out_size, block_size):
    """
    Expands the grid generated by the VAE (16x16 by default) to out_size x out_size.
    each pixel in the grid 16x16 becomes the constant value (= just DC) of a 8x8 block.

    Mathematically, doing upsampling-nearest neighbour
    is the same as doing IDCT in a block with just DC coefficients.
    """
    low_res = out_size // block_size
    dc_grid = F.adaptive_avg_pool2d(samples, output_size=(low_res, low_res))
    blocky = F.interpolate(dc_grid, size=(out_size, out_size), mode='nearest')
    return blocky.clamp(0.0, 1.0)


def generate_dataset():
    vae = load_vae(CHECKPOINT_PATH, DEVICE)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    generated = 0
    idx = 0
    while generated < NUM_SAMPLES:
        n = min(BATCH_SIZE, NUM_SAMPLES - generated)
        samples = vae_generate(n, vae, DEVICE, temperature=TEMPERATURE)     # [0,1], 16x16
        if DC_BLOCK_MODE:
            samples = to_dc_blocks(samples, OUT_SIZE, BLOCK_SIZE)
        else:
            samples = F.interpolate(samples, size=(OUT_SIZE, OUT_SIZE),
                                     mode='bilinear', align_corners=False)
        samples = samples.clamp(0.0, 1.0)
        samples_u8 = (samples * 255.0).round().clamp(0, 255).byte().cpu()

        for i in range(n):
            arr = samples_u8[i].permute(1, 2, 0).numpy()
            Image.fromarray(arr).save(os.path.join(OUTPUT_DIR, f"vae_face_{idx:06d}.jpg"))
            idx += 1

        generated += n
        print(f"{generated}/{NUM_SAMPLES} volti generati")

    print(f"Fatto. {NUM_SAMPLES} volti sintetici in {OUTPUT_DIR}")


In [ ]:
generate_dataset()

In [ ]:
# Get a list of all files in the output directory
image_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(('.jpg', '.png'))]
image_files.sort() # Ensure consistent order

if image_files:
    num_images_to_plot = min(6, len(image_files))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10)) # Create a 2x3 grid for up to 6 images
    axes = axes.flatten() # Flatten the 2x3 array of axes for easier iteration

    for i in range(num_images_to_plot):
        image_name = image_files[i]
        current_image_path = os.path.join(OUTPUT_DIR, image_name)

        # Load the image
        img = Image.open(current_image_path)

        # Plot the image on the current subplot
        axes[i].imshow(img)
        axes[i].set_title(f"Generated Image: {image_name}")
        axes[i].axis('off') # Hide axes ticks and labels

    # Turn off any unused subplots
    for j in range(num_images_to_plot, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()
else:
    print(f"No images found in {OUTPUT_DIR}. Please run generate_dataset() first.")